# Juan David Diaz V.
# Programming Exercise \#2
---

# Preamble

In [578]:
# optional code cell when using Google Colab with Google Drive

# remove the docstring comment block below in order to mount Google Drive
r'''
# mount Google Drive in Google Colab
from google.colab import drive
drive.mount('/content/drive')

# change directory using the magic command %cd
### replace [MY PATH] below with your own path in Google Drive ###
### %cd /content/drive/My\ Drive/[MY PATH] ###
'''

"\n# mount Google Drive in Google Colab\nfrom google.colab import drive\ndrive.mount('/content/drive')\n\n# change directory using the magic command %cd\n### replace [MY PATH] below with your own path in Google Drive ###\n### %cd /content/drive/My\\ Drive/[MY PATH] ###\n"

In [579]:
# import relevant Python libraries

import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
from scipy import stats as sps 
from IPython.display import display, Latex 

# **1. Environmental Sensor Telemetry**

## **Exploration**

In [580]:
telemetry_raw = pd.read_csv('iot_telemetry_dataset.csv') 
telemetry = telemetry_raw.copy() 
measurement_cols = ['co', 'humidity', 'lpg', 'smoke', 'temp'] 
print(telemetry.axes)
print(telemetry.dtypes)
telemetry.head(10)

[RangeIndex(start=0, stop=405184, step=1), Index(['ts', 'device', 'co', 'humidity', 'light', 'lpg', 'motion', 'smoke', 'temp'], dtype='str')]
ts          float64
device          str
co          float64
humidity    float64
light          bool
lpg         float64
motion         bool
smoke       float64
temp        float64
dtype: object


,ts,device,co,humidity,light,lpg,motion,smoke,temp
0,1.594512e+09,b8:27:eb:bf:9d:51,0.004956,51.000000,False,0.007651,False,0.020411,22.700000
1,1.594512e+09,00:0f:00:70:91:0a,0.002840,76.000000,False,0.005114,False,0.013275,19.700001
2,1.594512e+09,b8:27:eb:bf:9d:51,0.004976,50.900000,False,0.007673,False,0.020475,22.600000
3,1.594512e+09,1c:bf:ce:15:ec:4d,0.004403,76.800003,True,0.007023,False,0.018628,27.000000
4,1.594512e+09,b8:27:eb:bf:9d:51,0.004967,50.900000,False,0.007664,False,0.020448,22.600000
5,1.594512e+09,1c:bf:ce:15:ec:4d,0.004391,77.900002,True,0.007009,False,0.018589,27.000000
6,1.594512e+09,b8:27:eb:bf:9d:51,0.004976,50.900000,False,0.007673,False,0.020475,22.600000
7,1.594512e+09,00:0f:00:70:91:0a,0.002938,76.000000,False,0.005241,False,0.013628,19.700001
8,1.594512e+09,1c:bf:ce:15:ec:4d,0.004345,77.900002,True,0.006957,False,0.018440,27.000000
9,1.594512e+09,b8:27:eb:bf:9d:51,0.004970,50.900000,False,0.007667,False,0.020457,22.600000


### Problem 1.1

#### (a)

1. Motion is a binary/categorical varibale since it is based off its values tue or falso (0 or 1). This learning task is binary classification.
The learning setting is the supervision available. since the dataset contians labeld historicalobservations (we have both input features such as temperature/humitity and their corrspoinding ground truth "Motion" lables), this is a supervised learning setting.
2. If motion = false, it does not mean there is no person nearby necessarily. It means if there is a person and the motion = false, it is because the person hs not moved. PIR (passive infrared) only detecet changes in position not the static presence of a body or object. However, depending how sensitive the sensor is it can me more or less prone to detecting these changes in position (movement/motion).

#### (b)

In [581]:
print("Rows = Samples: ", telemetry.shape[0])
# print("Columns: ", telemetry.shape[1])

Rows = Samples:  405184


#### (c)

In [582]:
Sample_device1= (telemetry['device'] == '00:0f:00:70:91:0a').sum();
print("Number of samples from device 00:0f:00:70:91:0a = ", Sample_device1)

Number of samples from device 00:0f:00:70:91:0a =  111815


#### (d)

In [583]:
Sample_device2= (telemetry['device'] == '1c:bf:ce:15:ec:4d').sum();
print("Number of samples from device 1c:bf:ce:15:ec:4d = ", Sample_device2)

Number of samples from device 1c:bf:ce:15:ec:4d =  105918


#### (e)

In [584]:
Sample_device3= (telemetry['device'] == 'b8:27:eb:bf:9d:51').sum();
print("Number of samples from device b8:27:eb:bf:9d:51 = ", Sample_device3)

Number of samples from device b8:27:eb:bf:9d:51 =  187451


## **Preprocessing and Feature Construction**

### Problem 1.2

In [585]:
#Specify numerical measurement columns
measurement_cols = ['co', 'humidity', 'lpg', 'smoke', 'temp']

# 2. Standardize numerical measurements using scipy.stats.zscore
telemetry[measurement_cols] = sps.zscore(telemetry[measurement_cols])

# 3. Encode categorical / boolean variables
# Convert boolean columns light and motion to integers (0 and 1)
telemetry['light'] = telemetry['light'].astype(int)
telemetry['motion'] = telemetry['motion'].astype(int)

#### (a)

In [586]:

# Compute means and sample variances grouped by device
device_means = telemetry_raw.groupby('device')['co', 'humidity', 'lpg', 'smoke', 'temp'].mean()
device_vars = telemetry_raw.groupby('device')['co', 'humidity', 'lpg', 'smoke', 'temp'].var(ddof=1) #ddof stands for Delta Degrees of Freedom,
# set to 1 for sample variance.

# --- CHART 1: Grouped Bar Chart for Means ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Subpanel 1: Large/Medium scale measurements (humidity, temp)
device_means[['humidity', 'temp']].plot(kind='bar', ax=ax1, width=0.6)
ax1.set_title('Mean Measurements: Humidity & Temperature')
ax1.set_ylabel('Measurement Value')
ax1.set_xlabel('Device MAC Address')
ax1.tick_params(axis='x', rotation=15)
ax1.grid(axis='y', linestyle='--', alpha=0.7)

# Subpanel 2: Small scale measurements (co, lpg, smoke)
device_means[['co', 'lpg', 'smoke']].plot(kind='bar', ax=ax2, width=0.6)
ax2.set_title('Mean Measurements: CO, LPG, & Smoke')
ax2.set_ylabel('Measurement Value')
ax2.set_xlabel('Device MAC Address')
ax2.tick_params(axis='x', rotation=15)
ax2.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()


# --- CHART 2: Grouped Bar Chart for Sample Variances ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Subpanel 1: Large/Medium scale sample variance (humidity, temp)
device_vars[['humidity', 'temp']].plot(kind='bar', ax=ax1, width=0.6)
ax1.set_title('Sample Variance: Humidity & Temperature')
ax1.set_ylabel('Sample Variance')
ax1.set_xlabel('Device MAC Address')
ax1.tick_params(axis='x', rotation=15)
ax1.grid(axis='y', linestyle='--', alpha=0.7)

# Subpanel 2: Small scale sample variance (co, lpg, smoke)
device_vars[['co', 'lpg', 'smoke']].plot(kind='bar', ax=ax2, width=0.6)
ax2.set_title('Sample Variance: CO, LPG, & Smoke')
ax2.set_ylabel('Sample Variance')
ax2.set_xlabel('Device MAC Address')
ax2.tick_params(axis='x', rotation=15)
ax2.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

ValueError: Cannot subset columns with a tuple with more than one element. Use a list instead.

1. Difference in Average Readings (Means): Temperature: Device 1c:bf:ce:15:ec:4d records the highest average temperature (approx. $26.03^\circ\text{C}$)
Device 00:0f:00:70:91:0a records the lowest average temperature (approx. $19.36^\circ\text{C}$).
Humidity: Device 00:0f:00:70:91:0a exhibits the highest mean humidity (approx. $75.44\%$)
Device b8:27:eb:bf:9d:51 records the lowest mean humidity (approx. $50.81\%$).
2. Difference in Measurement Variability (Sample Variances):Humidity Variance: Device 1c:bf:ce:15:ec:4d displays dramatically higher sample variance in humidity ($\approx 80.01$) compared to devices 00:0f:00:70:91:0a ($\approx 3.90$) and b8:27:eb:bf:9d:51 ($\approx 3.57$), indicating that its environment undergoes significantly larger humidity fluctuations.
Smoke Variance: Device 00:0f:00:70:91:0a shows substantially higher variance in smoke readings ($\approx 2.3 \times 10^{-5}$) compared to device 1c:bf:ce:15:ec:4d ($\approx 1.0 \times 10^{-6}$), reflecting higher instability or fluctuation in air quality measurements for that specific location.

#### (b)

In [ ]:
measurement_cols = ['co', 'humidity', 'lpg', 'smoke', 'temp']

telemetry_12b = telemetry_raw.copy()
telemetry_12b[measurement_cols] = telemetry_12b.groupby('device')[
    measurement_cols
].transform(lambda group: (group - group.mean()) / group.std(ddof=1))

means_per_device = telemetry_12b.groupby('device')[measurement_cols].mean()
vars_per_device = telemetry_12b.groupby('device')[measurement_cols].var(ddof=1)

print("--- Sample Means (Per Device) ---")
print(means_per_device.round(6))

print("\n--- Sample Variances (Per Device) ---")
print(vars_per_device.round(6))

--- Sample Means (Per Device) ---
                    co  humidity  lpg  smoke  temp
device                                            
00:0f:00:70:91:0a -0.0       0.0  0.0    0.0  -0.0
1c:bf:ce:15:ec:4d  0.0      -0.0 -0.0    0.0   0.0
b8:27:eb:bf:9d:51 -0.0      -0.0 -0.0    0.0   0.0

--- Sample Variances (Per Device) ---
                    co  humidity  lpg  smoke  temp
device                                            
00:0f:00:70:91:0a  1.0       1.0  1.0    1.0   1.0
1c:bf:ce:15:ec:4d  1.0       1.0  1.0    1.0   1.0
b8:27:eb:bf:9d:51  1.0       1.0  1.0    1.0   1.0


No, if each measurement were standardized using all three devices together (global standardization), its mean and sample variance within each individual device would not necessarily be 0 and 1.
When standardizing globally, each data point $x_{i,k}$ belonging to device $k$ is transformed using the overall dataset mean $\bar{x}_{\text{global}}$ and overall standard deviation $s_{\text{global}}$:$$z_{i,k} = \frac{x_{i,k} - \bar{x}_{\text{global}}}{s_{\text{global}}}$$ 
Taking the mean and sample variance of $z_{i,k}$ within device $k$ yields: 
$$\bar{z}_k = \frac{\bar{x}_k - \bar{x}_{\text{global}}}{s_{\text{global}}}$$ $$s_{z, k}^2 = \frac{s_k^2}{s_{\text{global}}^2}$$ Within-device mean $\bar{z}_k = 0$ if and only if the device's original mean $\bar{x}_k$ happens to equal the global mean $\bar{x}_{\text{global}}$. Within-device variance $s_{z, k}^2 = 1$ if and only if the device's original variance $s_k^2$ happens to equal the global variance $s_{\text{global}}^2$. Because different devices are placed in different environments with distinct baseline readings and noise levels, individual device means $\bar{x}_k$ and variances $s_k^2$ differ from the global pooled parameters. Consequently, global standardization leaves non-zero means and non-unit variances within individual sub-groups

In [ ]:
# One-hot encode the device column (dropping first category to avoid multicollinearity)
telemetry = pd.get_dummies(telemetry, columns=['device'], drop_first=True, dtype=int)

#### (d)

In [ ]:
print("Baseline state (0, 0) uniquely identifies device 00:0f:00:70:91:0a \n")
telemetry.head(20)

Baseline state (0, 0) uniquely identifies device 00:0f:00:70:91:0a 



,ts,co,humidity,light,lpg,motion,smoke,temp,device_1c:bf:ce:15:ec:4d,device_b8:27:eb:bf:9d:51
0,1.594512e+09,0.253670,-0.836820,0,0.286471,0,0.280867,0.091172,0,1
1,1.594512e+09,-1.438976,1.362631,0,-1.469927,0,-1.465637,-1.020621,0,0
2,1.594512e+09,0.269729,-0.845618,0,0.301986,0,0.296495,0.054112,0,1
3,1.594512e+09,-0.188651,1.433013,1,-0.148041,0,-0.155499,1.684742,1,0
4,1.594512e+09,0.262810,-0.845618,0,0.295303,0,0.289763,0.054112,0,1
5,1.594512e+09,-0.198269,1.529789,1,-0.157652,0,-0.165121,1.684742,1,0
6,1.594512e+09,0.269739,-0.845618,0,0.301996,0,0.296504,0.054112,0,1
7,1.594512e+09,-1.360556,1.362631,0,-1.381916,0,-1.379324,-1.020621,0,0
8,1.594512e+09,-0.234694,1.529789,1,-0.194114,0,-0.201616,1.684742,1,0
9,1.594512e+09,0.265124,-0.845618,0,0.297538,0,0.292015,0.054112,0,1


#### (e)

1. The ts feature represents a Unix epoch timestamp (the number of seconds elapsed since January 1, 1970 UTC). It provides critical context about each observation: Temporal Sequence & Ordering: It establishes the exact chronological order in which sensor telemetry readings were captured across devices.
Exact Date & Time: Converting ts to human-readable datetime formats yields the specific date, hour, minute, and second of each measurement (for instance, 1.594512e+09 corresponds to July 11/12, 2020 UTC).   
Time Intervals & Rate of Sampling: Computing differences between successive timestamps ($\Delta t = ts_{i+1} - ts_i$) reveals the sampling rate and detects any missing telemetry cycles or network outages.   

2. Example of an analysis using ts (Time-Series Anomaly Detection / Trend Analysis):
An analyst can aggregate sensor values over time windows (e.g., hourly or daily averages) to detect temporal patterns, such as diurnal cycles (e.g., temperatures dropping at night or spike in CO/smoke during daytime hours) or gradual sensor drift.

3. No, standardizing ts (e.g., using Z-score scaling) would NOT help time-series analyses, and can actually hinder them.
It destroys Physical & Intuitive Meaning: Standardizing ts converts timestamps into relative Z-scores (with mean $0$ and standard deviation $1$). This strips away intuitive time references like exact hours, days of the week, or seasonal timestamps necessary to interpret time-dependent behaviors.   Preserves Distance Linearity (No Analytical Gain): Z-score standardization is a linear transformation: $$z = \frac{ts - \mu}{\sigma}$$
Because time intervals are scaled linearly by a constant factor ($\frac{1}{\sigma}$), relative duration differences between events remain identical. Standardizing provides no statistical advantage for time-series models like autoregression (ARIMA), rolling statistics, or lag features.   
Feature Engineering is Superior to Scaling: Instead of standardizing continuous Unix timestamps, time-series algorithms benefit from extracting cyclical/categorical temporal features—such as hour_of_day, day_of_week, or continuous time-elapsed vectors ($t - t_0$)

### Problem 1.3

#### (a)

In [588]:
X = telemetry_raw[measurement_cols].values

# 2. Compute the six row-wise statistical features
arithmetic_mean = np.mean(X, axis=1)
geometric_mean_abs = sps.gmean(np.abs(X), axis=1)
harmonic_mean_abs = sps.hmean(np.abs(X), axis=1)
sample_variance = np.var(X, axis=1, ddof=1)  # ddof=1 for sample variance
sample_excess_kurtosis = sps.kurtosis(X, axis=1, fisher=True, bias=False)  # bias=False applies sample correction
sample_skewness = sps.skew(X, axis=1, bias=False)  # bias=False applies sample correction

# 3. Construct the feature DataFrame
summary_features_df = pd.DataFrame({
    'arithmetic_mean': arithmetic_mean,
    'geometric_mean_abs': geometric_mean_abs,
    'harmonic_mean_abs': harmonic_mean_abs,
    'sample_variance': sample_variance,
    'sample_excess_kurtosis': sample_excess_kurtosis,
    'sample_skewness': sample_skewness
})

# 4. Display the first 40 rows without truncation
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 40)
pd.set_option('display.width', 1000)

summary_features_df.head(40)

,arithmetic_mean,geometric_mean_abs,harmonic_mean_abs,sample_variance,sample_excess_kurtosis,sample_skewness
0,14.746604,0.245731,0.013105,507.244728,1.098697,1.406088
1,19.144246,0.195924,0.008025,1082.894866,3.567629,1.903137
2,14.706625,0.245914,0.013151,505.036607,1.114426,1.409094
3,20.766012,0.260278,0.011814,1117.776178,2.421507,1.665210
4,14.706616,0.245700,0.013131,505.036943,1.114424,1.409094
5,20.985998,0.260664,0.011786,1148.837513,2.488614,1.678743
6,14.706625,0.245914,0.013151,505.036606,1.114426,1.409094
7,19.144362,0.199271,0.008270,1082.889338,3.567650,1.903141
8,20.985949,0.259311,0.011678,1148.840107,2.488605,1.678741
9,14.706619,0.245771,0.013138,505.036831,1.114424,1.409094


#### (b)

1. Distance relative to the device's mean: A standardized reading of +2 corresponds to an original reading that is 2 sample standard deviations above that device's mean. 
A standardized reading of −2 corresponds to an original reading that is 2 sample standard deviations below that device's mean.

2. Ability to distinguish above vs. below after taking absolute values
No, you cannot tell which reading was above the mean and which was below it after taking absolute values.
Explanation: Standardization transforms a raw reading x using the device mean μ and device sample standard deviation σ:

$$z = \frac{x−μ}{\sigma}$$

For  z=+2, the absolute value is ∣+2∣=2.
For z=−2, the absolute value is ∣−2∣=2.
Taking the absolute value discards the algebraic sign (+ or −), which is the only piece of information that indicates the direction relative to the mean. Both +2 and −2 map to the exact same magnitude (2), leaving only the distance from the mean while completely destroying the directional context of whether the measurement was higher or lower than average.

#### (c)

Metric Selection & Ground Truth Comparison:

Compare each model's predicted motion labels ($0$ or $1$) against the ground-truth motion labels recorded in the unseen test dataset. For binary classification, evaluate the following metrics: 
Accuracy: The total percentage of correct predictions ($True\ Positives + True\ Negatives$) out of all test samples.
Precision, Recall, and $F_1$-Score: Essential if class distributions for motion are imbalanced. 
Precision measures how reliable positive motion predictions are, while Recall measures how many actual motion events were successfully detected. Confusion Matrix: Map out $True\ Positives$, $False\ Positives$, $True\ Negatives$, and $False\ Negatives$ for both models side-by-side.


Decision Logic for Model Selection. 

Higher Metrics Win: The feature set that yields higher accuracy, $F_1$-score, and lower misclassification rates on unseen test observations is the superior input set.Feature Utility Insight: If the classifier trained on the five standardized measurements performs better, the individual raw sensor channels hold critical directional or channel-specific information that summary statistics destroy. If the classifier trained on the six constructed statistical features performs better, row-wise aggregation (like variance, skewness, or kurtosis) successfully captures noise/fluctuation patterns while filtering out redundant sensor noise.

# **2. Sample Averages and Estimation**

## **Estimating Means and Probabilities**

### Problem 2.1

#### (a)

In [ ]:
### Your code for 2.1(a) goes here ###

_[Your answer for 2.1(a) goes here]_

#### (b)

In [ ]:
### Your code for 2.1(b) goes here ###

_[Your answer for 2.1(b) goes here]_

#### (c)

In [ ]:
### Your code for 2.1(c) goes here ###

_[Your answer for 2.1(c) goes here]_

# **3. Classifying Environments from Noise Measurements**

## **Fitting the Class Models**

In [ ]:
### noise_train = pd.read_csv('NoiseClassificationTrainingData.csv') ###
### noise_classes = np.array([0, 1, 2]) ###
### if not np.isin(noise_train['ClassLabel'], noise_classes).all(): ###
###     raise ValueError('Unexpected training label.') ###
### train_x = noise_train['NoiseLevel'].to_numpy(dtype=float) ###
### train_y = noise_train['ClassLabel'].to_numpy(dtype=int) ###

### Problem 3.1

#### (a)

In [ ]:
### Your code for 3.1(a) goes here ###

#### (b)

_[Your answer for 3.1(b) goes here]_

### Problem 3.2

#### (a)

In [ ]:
### Your code for 3.2(a) goes here ###

#### (b)

_[Your answer for 3.2(b) goes here]_

### Problem 3.3

In [ ]:
### Your code for 3.3 goes here ###

## **Classification and Evaluation**

### Problem 3.4

In [ ]:
### Your code for 3.4 goes here ###

### Problem 3.5

#### (a)

In [ ]:
### Your code for 3.5(a) goes here ###

#### (b)

In [ ]:
### Your code for 3.5(b) goes here ###

### Problem 3.6

#### (a)

_[Your answer for 3.6(a) goes here]_

#### (b)

_[Your answer for 3.6(b) goes here]_

#### (c)

_[Your answer for 3.6(c) goes here]_

#### (d)

_[Your answer for 3.6(d) goes here]_

## **Decision-Region Exploration**

### Problem 3.7

#### (a)

In [ ]:
### Your code for 3.7(a) goes here ###

_[Your answer for 3.7(a) goes here]_

#### (b)

In [ ]:
### Your code for 3.7(b) goes here ###

_[Your answer for 3.7(b) goes here]_

#### (c)

In [ ]:
### Your code for 3.7(c) goes here ###

_[Your answer for 3.7(c) goes here]_